In [1]:
import requests
import pandas as pd

In [3]:

def fetch_solar(name, longitude, latitude, start_date, end_date):

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "shortwave_radiation,sunshine_duration",
        "timezone": "Pacific/Auckland"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame({
        "time":                         data["hourly"]["time"],
        "shortwave_radiation_wm2":       data["hourly"]["shortwave_radiation"],
        "sunshine_duration_s":           data["hourly"]["sunshine_duration"]
    })

    df["time"] = pd.to_datetime(df["time"])
    print(df.head(10))
    print(f"\nShape: {df.shape}")
    print(f"Missing values: {df.isnull().sum().sum()}")

    df = df.rename(columns={
        "shortwave_radiation_wm2":  f"{name}_shortwave_wm2",
        "sunshine_duration_s":      f"{name}_sunshine_s"
    })
    return df

In [8]:
#             name,              longitude,  latitude,  start_date,   end_date
request =  [['Auckland'     , 174.7633, -36.8485, "2014-01-01", "2026-05-25"],
            ['Christchurch' , 172.6362, -43.5321, "2014-01-01", "2026-05-25"],
            ['Wellington'   , 174.7762, -41.2865, "2014-01-01", "2026-05-25"],
            ['Hamilton'     , 175.2793, -37.7870, "2014-01-01", "2026-05-25"],
            ['Tauranga'     , 176.1651, -37.6878, "2014-01-01", "2026-05-25"],
            ['Dunedin'      , 170.5028, -45.8788, "2014-01-01", "2026-05-25"]]

solar_data = pd.DataFrame()
for name, lon, lat, start, end in request:
    df = fetch_solar(name, lon, lat, start, end)

    if solar_data.empty:
        solar_data = df
    else:
        solar_data = solar_data.merge(df, on="time", how="outer")

print(solar_data.head(5))

                 time  shortwave_radiation_wm2  sunshine_duration_s
0 2014-01-01 00:00:00                      0.0                 0.00
1 2014-01-01 01:00:00                      0.0                 0.00
2 2014-01-01 02:00:00                      0.0                 0.00
3 2014-01-01 03:00:00                      0.0                 0.00
4 2014-01-01 04:00:00                      0.0                 0.00
5 2014-01-01 05:00:00                      0.0                 0.00
6 2014-01-01 06:00:00                     39.0              3016.92
7 2014-01-01 07:00:00                    206.0              3600.00
8 2014-01-01 08:00:00                    425.0              3600.00
9 2014-01-01 09:00:00                    597.0              3600.00

Shape: (108672, 3)
Missing values: 0
                 time  shortwave_radiation_wm2  sunshine_duration_s
0 2014-01-01 00:00:00                      0.0                 0.00
1 2014-01-01 01:00:00                      0.0                 0.00
2 2014-01-

In [9]:
solar_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108672 entries, 0 to 108671
Data columns (total 13 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   time                        108672 non-null  datetime64[ns]
 1   Auckland_shortwave_wm2      108672 non-null  float64       
 2   Auckland_sunshine_s         108672 non-null  float64       
 3   Christchurch_shortwave_wm2  108672 non-null  float64       
 4   Christchurch_sunshine_s     108672 non-null  float64       
 5   Wellington_shortwave_wm2    108672 non-null  float64       
 6   Wellington_sunshine_s       108672 non-null  float64       
 7   Hamilton_shortwave_wm2      108672 non-null  float64       
 8   Hamilton_sunshine_s         108672 non-null  float64       
 9   Tauranga_shortwave_wm2      108672 non-null  float64       
 10  Tauranga_sunshine_s         108672 non-null  float64       
 11  Dunedin_shortwave_wm2       108672 non-

In [10]:
solar_data.to_csv("Solar_data.csv", index=False)